# 09.6 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/09-nlp-genai/06-fine-tuning/solutions.ipynb)

In [1]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import os, logging, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, str(data_path("download.py").parent))

## Exercise 1

Logits (1.2, −0.3): $e^{1.2} = 3.320$, $e^{-0.3} = 0.741$. $p(\text{positive}) = 0.741 / 4.061 = 0.182$. Loss $= -\ln 0.182 = 1.70$. The model is fairly sure of the wrong answer, so the loss is large.

Logits (−1.0, 2.0): $e^{-1} = 0.368$, $e^{2} = 7.389$. $p(\text{positive}) = 7.389 / 7.757 = 0.953$. Loss $= -\ln 0.953 = 0.049$. Confident and right, so the loss is small.

In [2]:
for logits in [np.array([1.2, -0.3]), np.array([-1.0, 2.0])]:
    p = np.exp(logits) / np.exp(logits).sum()
    print(f"logits {logits}: P(positive) = {p[1]:.3f}, loss = {-np.log(p[1]):.3f}")

logits [ 1.2 -0.3]: P(positive) = 0.182, loss = 1.701
logits [-1.  2.]: P(positive) = 0.953, loss = 0.049


## Setup

In [3]:
import transformers
transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()

import torch, time
torch.manual_seed(42)
device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
print("training on", device)

training on mps


In [4]:
from download import load_reviews
from sklearn.model_selection import train_test_split

reviews = load_reviews()          # first run downloads one 117 MB file, then cached
train, test = train_test_split(reviews, test_size=0.2, random_state=42, stratify=reviews["label"])
print(f"{len(reviews):,} reviews ({reviews['label'].mean():.0%} positive): "
      f"train {len(train):,} / test {len(test):,}")
reviews[["text", "label"]].head()

10,000 reviews (50% positive): train 8,000 / test 2,000


,text,label
0,"Great Book, Must read! Ramon found the fabulou...",1
1,A leaker! This is the second generation of the...,0
2,Not For the Professional Cook. Henckels are we...,0
3,Arg! I'd like to see Robert Jordan put to the ...,0
4,"If I could give this minus 5 stars, I'd give i...",0


In [5]:
N_TRAIN = 2000          # on a slow laptop CPU, 1000 halves the training time
small = train.groupby("label").sample(N_TRAIN // 2, random_state=42).sample(frac=1, random_state=42)
print(f"fine-tuning on {len(small):,} reviews, testing on {len(test):,}")

fine-tuning on 2,000 reviews, testing on 2,000


In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

CHECKPOINT = "distilbert/distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

def batches(texts, size):
    for i in range(0, len(texts), size):
        yield i, tokenizer(list(texts[i:i + size]), truncation=True, max_length=128,
                           padding=True, return_tensors="pt").to(device)

def train_one_epoch(model, texts, labels, lr=5e-5, batch_size=16):
    optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)
    labels, losses = torch.tensor(labels), []
    model.train()
    for i, batch in batches(texts, batch_size):
        loss = model(**batch, labels=labels[i:i + batch_size].to(device)).loss
        loss.backward(); optimiser.step(); optimiser.zero_grad()
        losses.append(loss.item())
    return losses

def predict_proba(model, texts, batch_size=64):
    model.eval(); out = []
    with torch.no_grad():
        for _, batch in batches(texts, batch_size):
            out.append(torch.softmax(model(**batch).logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(out)

## Exercise 2

In [7]:
results = {}
for n in [500, 2000]:
    torch.manual_seed(42)
    subset = small.groupby("label").head(n // 2)
    m = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2).to(device)
    train_one_epoch(m, subset["text"].tolist(), subset["label"].tolist())
    results[n] = np.mean((predict_proba(m, test["text"].tolist()) > 0.5) == test["label"].to_numpy())
    print(f"{n:>5} reviews: {results[n]:.1%}")

  500 reviews: 88.9%


 2000 reviews: 90.6%


With only 500 labelled reviews (32 training steps), the fine-tuned model already scores 88.9%, just 1.7 points below 2,000 reviews. Compare TF-IDF on 500 reviews (78.0%, 09.2) and embeddings on 500 (81.7%, 09.3). Fine-tuning makes the most of little data, because the network already knows English and only has to learn what *positive* means here.

## Exercise 3

In [8]:
torch.manual_seed(42)
# "eager" attention: Apple GPUs' fast attention kernel refuses dropout when the body is frozen
frozen = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2,
                                                            attn_implementation="eager").to(device)
for name, p in frozen.named_parameters():
    p.requires_grad = "classifier" in name
trainable = sum(p.numel() for p in frozen.parameters() if p.requires_grad)
start = time.time()
train_one_epoch(frozen, small["text"].tolist(), small["label"].tolist(), lr=1e-3)
secs = time.time() - start
acc = np.mean((predict_proba(frozen, test["text"].tolist()) > 0.5) == test["label"].to_numpy())
print(f"head only: {trainable:,} trainable parameters, {secs:.0f} s, accuracy {acc:.1%}")

head only: 592,130 trainable parameters, 5 s, accuracy 86.4%


Training only the head is faster (no gradients flow back through the 66 million body weights) but about 4 points less accurate than full fine-tuning (86.4% against 90.6%). The body was pre-trained to fill in missing words, not to judge sentiment; its frozen output carries a decent sentiment signal, but letting all the weights adjust a little is what makes fine-tuning strong. (The head gets a higher learning rate because it starts from scratch.)

## Exercise 4

In [9]:
torch.manual_seed(42)
ft = AutoModelForSequenceClassification.from_pretrained(CHECKPOINT, num_labels=2).to(device)
train_one_epoch(ft, small["text"].tolist(), small["label"].tolist())
scored = test.assign(p_positive=predict_proba(ft, test["text"].tolist()))
scored["confidence_wrong"] = np.where(scored["label"] == 1, 1 - scored["p_positive"], scored["p_positive"])
pd.set_option("display.max_colwidth", 220)
scored.nlargest(5, "confidence_wrong")[["label", "p_positive", "text"]]

,label,p_positive,text
4256,0,0.994988,"Good Ones. I bought these for my kids toys, been using these for almost 2 months now.I am happy with the product."
7789,0,0.994150,Extremely funny. I like the way it was written (as a diary). I couldn't stop laughing. I can't wait to see the movie and read the sequel.
1971,0,0.992965,How did Brian Mcknight and Russell Crowe get on this CD. I couldnt wait to hear this CD cause i love Kris Kristofferson and his work. many of the names on the CD were great and some great singers came out to do triBu...
5815,0,0.991931,"Smelly. Be prepared for a very strong odor -- like rotten eggs with strong cheap perfume. If you are willing to put up with the stench, the results are amazing."
6582,0,0.988572,"Thai Fresh Lemongrass - 8 stalks. Exactly what I wanted, will reorder when I run out. Came fresh and makes the best tea. Thank you very much.."


Read them as a human would. All five are labelled negative and the model is over 98% sure they are positive. Three are plainly positive reviews with a wrong label (*"I am happy with the product"*, *"Extremely funny… I couldn't stop laughing"*, *"Exactly what I wanted, will reorder"*). The other two are genuinely mixed (*"Smelly… but the results are amazing"*). Past about 90% accuracy, a growing share of the remaining "errors" are in the data, not the model.